# Part 4 — Generative AI Follow-up Notes Analysis

CareLoop Clinics — No-Show Guard


In [1]:
!pip install -q google-genai pydantic

In [2]:
import json
import pandas as pd
import numpy as np

from google import genai
from google.genai import types
from getpass import getpass

In [3]:
MODEL_NAME = "gemini-3.5-flash-lite"

API_KEY = getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

Paste your Gemini API key: ··········
Gemini client initialized successfully!


In [4]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [5]:
SYSTEM_TEXT = """
You are a healthcare appointment follow-up analyst.

Your responsibility is to analyse patient follow-up notes
and identify the most likely reason for a missed appointment.

Use only the information explicitly supported by the note.
Do not invent details or make unsupported assumptions.

Return a structured response following the requested JSON schema.
Be concise, neutral and evidence-based.
"""

In [6]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

def build_prompt(note_text):

    prompt = f"""
CONTEXT:
A clinic wants to understand why patients miss appointments.

ROLE:
Act as a patient follow-up notes analyst.

ACTION:
Read the patient note and identify its most likely root cause.

Choose exactly one of these allowed categories:
{ALLOWED_CAUSES}

If the note is ambiguous or does not provide enough evidence,
select "other" rather than guessing.

Also provide:
1. Confidence between 0 and 1.
2. A summary containing at most 25 words.
3. A practical recommended action for the clinic.

TONE:
Professional, neutral, concise and evidence-based.

FORMAT:
Return all four requested fields using the supplied JSON schema.
Do not add explanations outside the JSON response.

PATIENT NOTE:
{note_text}
"""
    return prompt

In [7]:
def analyse_note(note_text):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1200,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    raw_text = response.text

    # Keep the original response for Task 4.2
    analyse_note.last_raw_response = raw_text

    # Convert JSON text into a Python dictionary
    data = json.loads(raw_text)

    return data


analyse_note.last_raw_response = ""

In [9]:
notes_df = pd.read_csv("followup_notes.csv")

print(notes_df.head(10))
print(notes_df.columns.tolist())

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...
['note_id', 'appointment_id', 'note_text']


In [10]:
n001_note = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

print(n001_note)

Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [11]:
result_N001 = analyse_note(n001_note)

print(result_N001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient did not receive a reminder message and forgot the appointment, requesting to reschedule for next week.', 'recommended_action': "Verify and update the patient's contact information in the system to ensure future reminder delivery."}


In [12]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)

In [13]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [14]:
validated = FollowupInsight(**result_N001)

if validated.root_cause not in ALLOWED_CAUSES:
    print("Invalid root cause")
else:
    print("Validation successful!")

print(validated.model_dump())

Validation successful!
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient did not receive a reminder message and forgot the appointment, requesting to reschedule for next week.', 'recommended_action': "Verify and update the patient's contact information in the system to ensure future reminder delivery."}


In [15]:
print(notes_df.columns.tolist())

['note_id', 'appointment_id', 'note_text']


In [16]:
import time

results = []
task42_api_calls = 0

for _, row in notes_df.head(10).iterrows():

    note_text = row["note_text"]

    # Check blank notes before calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":

        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

        print(row["note_id"], "Skipped: blank note")
        continue

    # Pause between Gemini calls
    if task42_api_calls > 0:
        time.sleep(13)

    data = {}
    raw_response = ""

    try:
        task42_api_calls += 1

        data = analyse_note(str(note_text))

        raw_response = analyse_note.last_raw_response

        # Pydantic validation
        validated = FollowupInsight(**data)

        # Allowed-category validation
        if validated.root_cause not in ALLOWED_CAUSES:
            raise ValueError(
                f"Invalid root cause: {validated.root_cause}"
            )

        status = "valid"

        print(row["note_id"], "Valid")

    except Exception as e:

        status = "invalid"

        print(row["note_id"], "Invalid:", str(e))

    # Preserve returned fields without repairing the response
    results.append({
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),
        "validation_status": status,
        "raw_response": raw_response
    })

print("Task 4.2 API calls:", task42_api_calls)

N001 Valid
N002 Valid
N003 Valid
N004 Skipped: blank note
N005 Valid
N006 Valid
N007 Valid
N008 Valid
N009 Valid
N010 Valid
Task 4.2 API calls: 9


In [17]:
notes_analysed = pd.DataFrame(results)

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print(notes_analysed)
print("\nShape:", notes_analysed.shape)

  note_id appointment_id             root_cause  confidence  \
0    N001          A0872  forgot_or_no_reminder        1.00   
1    N002          A0897  distance_or_transport        1.00   
2    N003          A0526            cost_or_fee        1.00   
3    N004          A0926                  other        0.00   
4    N005          A0534   illness_or_emergency        1.00   
5    N006          A0507       work_or_schedule        1.00   
6    N007          A0333  forgot_or_no_reminder        1.00   
7    N008          A0222  distance_or_transport        1.00   
8    N009          A0977                  other        0.50   
9    N010          A0596       work_or_schedule        0.95   

                                             summary  \
0  The patient stated she did not receive a remin...   
1  The patient missed the routine check-up becaus...   
2  The patient skipped the appointment because th...   
3                                   No note recorded   
4  Patient missed the appo

In [18]:
print(
    notes_analysed[
        notes_analysed["note_id"] == "N004"
    ]
)

  note_id appointment_id root_cause  confidence           summary  \
3    N004          A0926      other         0.0  No note recorded   

  recommended_action validation_status raw_response  
3    Re-call patient     skipped_blank               


In [19]:
from pydantic import ValidationError

# Test 1: Invalid confidence
bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Patient reason is unclear",
    "recommended_action": "Contact the patient"
}

try:
    FollowupInsight(**bad_reply_1)

except ValidationError as e:
    print("Test 1 Error:")
    print(e)


# Test 2: Missing recommended_action
bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.8,
    "summary": "Patient reason is unclear"
}

try:
    FollowupInsight(**bad_reply_2)

except ValidationError as e:
    print("\nTest 2 Error:")
    print(e)

Test 1 Error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2 Error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...ient reason is unclear'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [20]:
comparison_results = []

for note_id in ["N007", "N009"]:

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id,
        "note_text"
    ].iloc[0]

    prompt_v1 = f"Classify this note:\n{note_text}"

    try:
        time.sleep(13)

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_v1,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_TEXT,
                temperature=0,
                max_output_tokens=1200,
                response_mime_type="application/json",
                response_schema=SCHEMA
            )
        )

        v1_data = json.loads(response.text)
        v1_cause = v1_data.get("root_cause", "")

        comparison_results.append({
            "note_id": note_id,
            "V1_simple": v1_cause,
            "V1_valid": v1_cause in ALLOWED_CAUSES
        })

    except Exception as e:
        print(note_id, "V1 failed:", e)

In [21]:
for note_id in ["N007", "N009"]:

    row = notes_analysed[
        notes_analysed["note_id"] == note_id
    ].iloc[0]

    v2_cause = row["root_cause"]

    for result in comparison_results:
        if result["note_id"] == note_id:
            result["V2_CRAFT"] = v2_cause
            result["V2_valid"] = v2_cause in ALLOWED_CAUSES

comparison_table = pd.DataFrame(comparison_results)

print(comparison_table)

  note_id                            V1_simple  V1_valid  \
0    N007  forgot and no SMS reminder received     False   
1    N009                          unreachable     False   

                V2_CRAFT  V2_valid  
0  forgot_or_no_reminder      True  
1                  other      True  


In [22]:
comparison_table_final = pd.DataFrame([
    {
        "note_id": "N007",
        "prompt": "V1 - Simple",
        "root_cause": "",
        "status": "503 UNAVAILABLE"
    },
    {
        "note_id": "N007",
        "prompt": "V2 - CRAFT",
        "root_cause": "forgot_or_no_reminder",
        "status": "allowed"
    },
    {
        "note_id": "N009",
        "prompt": "V1 - Simple",
        "root_cause": "unreachable",
        "status": "not allowed"
    },
    {
        "note_id": "N009",
        "prompt": "V2 - CRAFT",
        "root_cause": "other",
        "status": "allowed"
    }
])

print(comparison_table_final)

  note_id       prompt             root_cause           status
0    N007  V1 - Simple                         503 UNAVAILABLE
1    N007   V2 - CRAFT  forgot_or_no_reminder          allowed
2    N009  V1 - Simple            unreachable      not allowed
3    N009   V2 - CRAFT                  other          allowed


###I would deploy V2 (CRAFT) because it provides context, defines the permitted root-cause categories and instructs Gemini to select other when evidence is insufficient. The simple prompt produced an invalid category for N009, whereas CRAFT returned an allowed category for both notes. N007's V1 request was unsuccessful because of a temporary API availability error

## Task 4.4 — Merge GenAI insights with appointment data

Join the ten analysed follow-up notes to the cleaned appointment table using `appointment_id`, inspect reminder status by root cause, and save the merged output.

In [24]:
appointments_clean = pd.read_csv("appointments_clean.csv")
notes_analysed = pd.read_csv("notes_analysed.csv")

notes_merged = notes_analysed.merge(
    appointments_clean[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", notes_merged.shape)
print("\nRoot cause by reminder status:")
print(pd.crosstab(notes_merged["root_cause"], notes_merged["reminder_sent"]))

notes_merged.to_csv("notes_merged.csv", index=False)
print("\nSaved notes_merged.csv")

Merged shape: (10, 9)

Root cause by reminder status:
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Saved notes_merged.csv
